# Voxelizer & Geodesic Reward Visualization

Tests the voxel distance field output and simulates how the reward lookup works per-drone per-env.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json

VOXEL_DIR = "voxel_output"

with open(f"{VOXEL_DIR}/grid_metadata.json") as f:
    meta = json.load(f)

resolution   = meta["resolution"]          # 0.05 m
origin       = np.array(meta["origin"])    # [-5.55, -5.55, -0.5]
grid_shape   = meta["grid_shape"]          # [223, 222, 220]
goal_world   = np.array(meta["goal_world"]) # [-4.5, 3.5, 7.0]
goal_voxel   = meta["goal_voxel"]          # [21, 181, 150]
max_geo_dist = meta["max_geodesic_dist_m"] # 29.35

print("Resolution:", resolution, "m")
print("Origin:", origin)
print("Grid shape (X,Y,Z):", grid_shape)
print("Goal world:", goal_world, "  Goal voxel:", goal_voxel)
print("Max geodesic dist:", max_geo_dist, "m")

In [ ]:
data        = np.load(f"{VOXEL_DIR}/distance_field.npz")
occupancy   = data["occupancy"]        # bool (223,222,220)
dist_field  = data["distance_field"]   # float32 (223,222,220), geodesic dist in meters

print("Occupancy shape:", occupancy.shape, "  dtype:", occupancy.dtype)
print("Distance field shape:", dist_field.shape, "  dtype:", dist_field.dtype)
print(f"Occupied voxels: {occupancy.sum():,} / {occupancy.size:,} ({100*occupancy.mean():.1f}%)")
print(f"Dist field — min: {dist_field[~occupancy].min():.2f} m  max: {dist_field[~occupancy].max():.2f} m")

gv = goal_voxel
goal_occ = occupancy[gv[0], gv[1], gv[2]]
goal_dist = dist_field[gv[0], gv[1], gv[2]]
print()
print(f"=== GOAL SANITY CHECK ===")
print(f"Requested goal world: {goal_world}")
print(f"Goal voxel {gv}  occupied: {goal_occ}  dist: {goal_dist:.4f}")
if goal_occ:
    print("  *** GOAL IS INSIDE AN OBSTACLE — voxelizer snapped to nearest free voxel ***")
    print("  *** The distance field may be anchored OUTSIDE the warehouse! ***")
    print("  *** Re-run voxelizer with a goal inside free space. ***")
elif goal_dist != 0.0:
    print(f"  *** WARNING: goal voxel dist={goal_dist} (not 0) — BFS may have snapped the start! ***")
else:
    print("  Goal voxel is FREE and dist=0. ✓")

## Helper: world ↔ voxel coordinate conversion

In [ ]:
def world_to_voxel(pos_world):
    """Convert world-local position (relative to env origin) to voxel indices."""
    idx = np.floor((np.array(pos_world) - origin) / resolution).astype(int)
    idx = np.clip(idx, 0, np.array(grid_shape) - 1)
    return idx

def voxel_to_world(idx):
    return np.array(idx) * resolution + origin

def query_geo_dist(pos_world):
    idx = world_to_voxel(pos_world)
    return dist_field[idx[0], idx[1], idx[2]]

def euclidean_dist(pos_world):
    return np.linalg.norm(np.array(pos_world) - goal_world)

# Sanity checks
spawn_pos = [3.5, 4.0, 5.0]  # drone spawn (env-local)
print(f"Spawn pos {spawn_pos}")
print(f"  → voxel idx:     {world_to_voxel(spawn_pos)}")
print(f"  → geo dist:      {query_geo_dist(spawn_pos):.2f} m")
print(f"  → euclidean dist:{euclidean_dist(spawn_pos):.2f} m")
print()
print(f"Goal pos {goal_world}")
print(f"  → voxel idx:     {world_to_voxel(goal_world)}")
print(f"  → geo dist:      {query_geo_dist(goal_world):.4f} m  (should be ~0)")

## Reward function simulation

In [ ]:
def reward_euclid(pos_world, scale=15.0, step_dt=0.02):
    d = euclidean_dist(pos_world)
    return (1.0 - np.tanh(d / 0.8)) * scale * step_dt

def reward_geodesic(pos_world, scale=15.0, step_dt=0.02):
    d = query_geo_dist(pos_world)
    return (1.0 - np.tanh(d / 0.8)) * scale * step_dt

test_positions = [
    ("spawn",         [3.5,  4.0,  5.0]),
    ("near goal",     [-4.3, 3.4,  6.9]),
    ("mid-path",      [0.0,  0.0,  5.0]),
    ("opposite side", [4.5, -4.5,  5.0]),
]

print(f"{'Position':<16} {'Euclid(m)':>10} {'Geodesic(m)':>12} {'R_euclid':>10} {'R_geodesic':>11} {'Ratio geo/euc':>14}")
print("-" * 78)
for name, pos in test_positions:
    ed = euclidean_dist(pos)
    gd = query_geo_dist(pos)
    re = reward_euclid(pos)
    rg = reward_geodesic(pos)
    print(f"{name:<16} {ed:>10.2f} {gd:>12.2f} {re:>10.4f} {rg:>11.4f} {gd/ed if ed>0 else 0:>14.2f}x")

## 2D slice visualization: occupancy + distance field at drone flight height

In [ ]:
flight_z_world = 5.0  # meters, typical drone flight height
z_idx = int((flight_z_world - origin[2]) / resolution)
z_idx = np.clip(z_idx, 0, grid_shape[2] - 1)
print(f"Flight height {flight_z_world}m → Z voxel index {z_idx}")

occ_slice  = occupancy[:, :, z_idx]    # (X, Y)
dist_slice = dist_field[:, :, z_idx]   # (X, Y)

# Mask occupied voxels out of distance display
dist_masked = np.where(occ_slice, np.nan, dist_slice)

# Axis tick labels in world coords
x_world = origin[0] + np.arange(grid_shape[0]) * resolution
y_world = origin[1] + np.arange(grid_shape[1]) * resolution

goal_xi = goal_voxel[0]
goal_yi = goal_voxel[1]
spawn_vi = world_to_voxel(spawn_pos)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Occupancy ---
ax = axes[0]
ax.imshow(occ_slice.T, origin="lower", cmap="Greys",
          extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*", ms=14, label="goal")
ax.plot(spawn_pos[0], spawn_pos[1],   "b^", ms=10, label="spawn")
ax.set_title(f"Occupancy slice  z={flight_z_world}m")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend()

# --- Geodesic distance ---
ax = axes[1]
im = ax.imshow(dist_masked.T, origin="lower", cmap="viridis",
               extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.imshow(occ_slice.T, origin="lower", cmap="Greys", alpha=0.3,
          extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*", ms=14, label="goal")
ax.plot(spawn_pos[0], spawn_pos[1],   "b^", ms=10, label="spawn")
plt.colorbar(im, ax=ax, label="Geodesic dist (m)")
ax.set_title(f"Geodesic distance field  z={flight_z_world}m")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend()

plt.tight_layout()
plt.show()

## Euclidean vs Geodesic reward along a straight-line trajectory

In [ ]:
# Simulate a drone flying straight from spawn to goal (ignores walls)
n_steps = 200
t = np.linspace(0, 1, n_steps)
traj = np.array(spawn_pos) + t[:, None] * (goal_world - np.array(spawn_pos))

euclid_dists = np.array([euclidean_dist(p) for p in traj])
geo_dists    = np.array([query_geo_dist(p) for p in traj])
r_euclid     = (1.0 - np.tanh(euclid_dists / 0.8)) * 15.0 * 0.02
r_geodesic   = (1.0 - np.tanh(geo_dists    / 0.8)) * 15.0 * 0.02

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax = axes[0]
ax.plot(t, euclid_dists, label="Euclidean dist", color="steelblue")
ax.plot(t, geo_dists,    label="Geodesic dist",  color="darkorange")
ax.set_ylabel("Distance to goal (m)")
ax.set_title("Straight-line trajectory: Euclidean vs Geodesic distance")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(t, r_euclid,   label="Reward (euclidean)", color="steelblue")
ax.plot(t, r_geodesic, label="Reward (geodesic)",  color="darkorange")
ax.set_xlabel("Trajectory progress (0=spawn, 1=goal)")
ax.set_ylabel("Reward per step")
ax.set_title("Reward comparison along straight-line path")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Geo dist at spawn: {geo_dists[0]:.2f}m   Euclid: {euclid_dists[0]:.2f}m")
print(f"Geo dist at goal:  {geo_dists[-1]:.4f}m  Euclid: {euclid_dists[-1]:.4f}m")

## Vectorized reward lookup — simulate N envs (mirrors training code)

In [ ]:
import torch

device = "cpu"

dist_field_t  = torch.tensor(dist_field,  dtype=torch.float32, device=device)
grid_origin_t = torch.tensor(origin,      dtype=torch.float32, device=device)
grid_res      = float(resolution)
grid_shape_l  = list(dist_field.shape)

def query_distance_field_batch(pos_w, env_origins):
    """Mirrors _query_distance_field in obstacle_nav_env.py."""
    local = pos_w - env_origins
    idx   = ((local - grid_origin_t) / grid_res).long()
    idx[:, 0].clamp_(0, grid_shape_l[0] - 1)
    idx[:, 1].clamp_(0, grid_shape_l[1] - 1)
    idx[:, 2].clamp_(0, grid_shape_l[2] - 1)
    return dist_field_t[idx[:, 0], idx[:, 1], idx[:, 2]]

# Simulate 16 envs, each with a different origin (grid layout, 11m spacing)
N = 16
env_origins = torch.zeros(N, 3, device=device)
env_origins[:, 0] = (torch.arange(N) % 4) * 11.0
env_origins[:, 1] = (torch.arange(N) // 4) * 11.0

# All drones at spawn position (relative to env origin)
spawn_t  = torch.tensor(spawn_pos, dtype=torch.float32, device=device)
pos_w    = env_origins + spawn_t.unsqueeze(0)

geo_dists_batch = query_distance_field_batch(pos_w, env_origins)
print(f"Geo dist at spawn for {N} envs (should all be equal):")
print(geo_dists_batch.numpy())
print(f"Mean: {geo_dists_batch.mean():.4f}  Std: {geo_dists_batch.std():.6f}  (std should be ~0)")

## Vertical profile: geodesic distance vs height at spawn XY

In [ ]:
z_range  = np.linspace(origin[2], origin[2] + grid_shape[2]*resolution - resolution, grid_shape[2])
col_xy   = [spawn_pos[0], spawn_pos[1]]
geo_z    = np.array([query_geo_dist([col_xy[0], col_xy[1], z]) for z in z_range])
occ_z    = np.array([occupancy[world_to_voxel([col_xy[0], col_xy[1], z])[0],
                                world_to_voxel([col_xy[0], col_xy[1], z])[1],
                                world_to_voxel([col_xy[0], col_xy[1], z])[2]] for z in z_range])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_range[~occ_z], geo_z[~occ_z], color="darkorange", lw=1.5, label="geodesic dist (free)")
ax.scatter(z_range[occ_z], [0]*occ_z.sum(), color="gray", s=5, label="occupied", zorder=3)
ax.axvline(spawn_pos[2],  color="blue",  ls="--", label=f"spawn z={spawn_pos[2]}m")
ax.axvline(goal_world[2], color="red",   ls="--", label=f"goal z={goal_world[2]}m")
ax.set_xlabel("Z (m)")
ax.set_ylabel("Geodesic dist to goal (m)")
ax.set_title(f"Geodesic distance vs height at spawn XY ({col_xy})")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## BFS Path Tracing: spawn → goal

Reconstruct the navigable path by greedily descending the distance field (always step to the free 26-connected neighbor with the lowest geodesic distance). This is the exact path the BFS "knows about" — if it renders correctly through the warehouse it confirms the field is valid.

In [ ]:
from itertools import product as iproduct

def trace_bfs_path(start_world, max_steps=5000):
    """
    Greedy gradient descent on dist_field from start to goal.
    At each step: pick the free 26-neighbor with smallest dist value.
    Returns (path_voxels, path_world).
    """
    cur = world_to_voxel(start_world).tolist()
    path_voxels = [cur[:]]
    offsets = [list(o) for o in iproduct([-1,0,1], repeat=3) if any(v != 0 for v in o)]

    for _ in range(max_steps):
        xi, yi, zi = cur
        if dist_field[xi, yi, zi] < resolution:
            break

        best_dist = dist_field[xi, yi, zi]
        best_nb   = None
        for dx, dy, dz in offsets:
            nx, ny, nz = xi+dx, yi+dy, zi+dz
            if not (0 <= nx < grid_shape[0] and 0 <= ny < grid_shape[1] and 0 <= nz < grid_shape[2]):
                continue
            if occupancy[nx, ny, nz]:
                continue
            d = dist_field[nx, ny, nz]
            if d < best_dist:
                best_dist = d
                best_nb   = [nx, ny, nz]

        if best_nb is None:
            print("WARNING: stuck — no improving free neighbor found at", cur)
            break
        cur = best_nb
        path_voxels.append(cur[:])

    path_world = np.array([voxel_to_world(v) for v in path_voxels])
    return path_voxels, path_world

spawn_pos = [3.5, 4.0, 5.0]
path_voxels, path_world = trace_bfs_path(spawn_pos)

print(f"Path length: {len(path_voxels)} voxel steps")
print(f"Path distance (sum of steps): {len(path_voxels) * resolution:.2f} m")
print(f"Geodesic dist at spawn:       {query_geo_dist(spawn_pos):.2f} m")
print(f"Start: {path_world[0]}  →  End: {path_world[-1]}")
print(f"Goal:  {goal_world}")
print(f"Final distance to goal: {np.linalg.norm(path_world[-1] - goal_world):.3f} m")

In [ ]:
x_world = origin[0] + np.arange(grid_shape[0]) * resolution
y_world = origin[1] + np.arange(grid_shape[1]) * resolution
z_world = origin[2] + np.arange(grid_shape[2]) * resolution

# Project occupancy ONLY over the Z range the path actually uses.
# Using any(axis=2) over ALL heights makes floor-level walls appear
# even when the drone flies well above them — that's the "path through obstacles" illusion.
path_z_voxels = [v[2] for v in path_voxels]
z_lo, z_hi = min(path_z_voxels), max(path_z_voxels)
occ_xy = occupancy[:, :, z_lo:z_hi+1].any(axis=2)   # top-down, path height range only
occ_xz = occupancy[:, :, :].any(axis=1)              # side view: collapse Y (full range is fine)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# XY top-down — obstacles at path flight altitude
ax = axes[0]
ax.imshow(occ_xy.T, origin="lower", cmap="Greys", alpha=0.5,
          extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.plot(path_world[:, 0], path_world[:, 1], ".-", color="limegreen", lw=1.5, ms=2, label="BFS path")
ax.plot(path_world[0, 0], path_world[0, 1], "b^", ms=12, zorder=5, label="spawn")
ax.plot(goal_world[0],    goal_world[1],     "r*", ms=14, zorder=5, label="goal")
ax.set_title(f"BFS path — top-down XY\n(obstacles projected over path Z range "
             f"{z_world[z_lo]:.1f}–{z_world[z_hi]:.1f} m)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend(); ax.grid(True, alpha=0.2)

# XZ side view
ax = axes[1]
ax.imshow(occ_xz.T, origin="lower", cmap="Greys", alpha=0.5,
          extent=[x_world[0], x_world[-1], z_world[0], z_world[-1]], aspect="equal")
ax.plot(path_world[:, 0], path_world[:, 2], ".-", color="limegreen", lw=1.5, ms=2, label="BFS path")
ax.plot(path_world[0, 0], path_world[0, 2], "b^", ms=12, zorder=5, label="spawn")
ax.plot(goal_world[0],    goal_world[2],     "r*", ms=14, zorder=5, label="goal")
ax.set_title("BFS path — side XZ view")
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)")
ax.legend(); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# Quick collision check: are any path voxels inside occupied cells?
collisions = [v for v in path_voxels if occupancy[v[0], v[1], v[2]]]
print(f"Path voxels in collision: {len(collisions)} (should be 0)")

In [ ]:
import plotly.graph_objects as go
from scipy.ndimage import binary_erosion

# Extract surface voxels: occupied AND has at least one free 6-connected neighbor
# binary_erosion shrinks the solid by 1 voxel — difference = surface shell
interior = binary_erosion(occupancy, border_value=1)
surface  = occupancy & ~interior

surf_idx   = np.argwhere(surface)                       # (N, 3) voxel indices
surf_world = surf_idx * resolution + origin             # world coords

print(f"Total occupied voxels : {occupancy.sum():,}")
print(f"Surface voxels        : {surface.sum():,}  ({100*surface.sum()/occupancy.sum():.1f}% of occupied)")

# Colour surface voxels by height (Z) for depth cue
z_vals  = surf_world[:, 2]
z_norm  = (z_vals - z_vals.min()) / (z_vals.ptp() + 1e-9)

fig = go.Figure()

# Obstacles (surface shell)
fig.add_trace(go.Scatter3d(
    x=surf_world[:, 0], y=surf_world[:, 1], z=surf_world[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=z_norm, colorscale="Greys",
                opacity=0.4, colorbar=dict(title="Z (norm)")),
    name="obstacles",
))

# BFS path
fig.add_trace(go.Scatter3d(
    x=path_world[:, 0], y=path_world[:, 1], z=path_world[:, 2],
    mode="lines",
    line=dict(color=np.linspace(0, 1, len(path_world)), colorscale="Plasma", width=4),
    name="BFS path",
))

# Spawn & goal
fig.add_trace(go.Scatter3d(
    x=[path_world[0, 0]], y=[path_world[0, 1]], z=[path_world[0, 2]],
    mode="markers", marker=dict(size=8, color="blue", symbol="diamond"),
    name="spawn",
))
fig.add_trace(go.Scatter3d(
    x=[goal_world[0]], y=[goal_world[1]], z=[goal_world[2]],
    mode="markers", marker=dict(size=10, color="red", symbol="cross"),
    name="goal",
))

fig.update_layout(
    title="Voxelized warehouse — interactive 3D (drag to rotate, scroll to zoom)",
    scene=dict(
        xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
        aspectmode="data",   # preserves real-world proportions
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(itemsizing="constant"),
)

fig.show()

## Interactive 3D environment viewer (Plotly)

Full voxelized environment with free rotation/zoom. Only surface voxels are shown (occupied voxels with at least one free neighbor) to keep it fast.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa

# --- Subsample occupied voxels for 3D display (full grid is too heavy) ---
occ_idx = np.argwhere(occupancy)  # (N, 3) voxel indices of all obstacles
occ_world = occ_idx * resolution + origin  # world coords

# Only keep voxels within the Z range the path actually uses (+ small margin)
z_lo_w = z_world[z_lo] - 0.5
z_hi_w = z_world[z_hi] + 0.5
mask = (occ_world[:, 2] >= z_lo_w) & (occ_world[:, 2] <= z_hi_w)
occ_world_filt = occ_world[mask]

# Subsample to at most 8000 points so matplotlib stays responsive
rng = np.random.default_rng(42)
if len(occ_world_filt) > 8000:
    idx = rng.choice(len(occ_world_filt), 8000, replace=False)
    occ_world_filt = occ_world_filt[idx]

print(f"Obstacle points shown: {len(occ_world_filt):,}  (Z={z_lo_w:.1f}–{z_hi_w:.1f} m)")

# --- Plot ---
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

# Obstacles — small grey dots
ax.scatter(occ_world_filt[:, 0], occ_world_filt[:, 1], occ_world_filt[:, 2],
           c="grey", s=1, alpha=0.15, label="obstacles")

# Path — colored by progress (blue→red) so you can see the Z changes
n = len(path_world)
colors = plt.cm.plasma(np.linspace(0, 1, n))
for i in range(n - 1):
    ax.plot(path_world[i:i+2, 0], path_world[i:i+2, 1], path_world[i:i+2, 2],
            color=colors[i], lw=1.5)

# Markers
ax.scatter(*path_world[0],  color="blue", s=120, zorder=5, label="spawn")
ax.scatter(*goal_world,     color="red",  s=200, marker="*", zorder=5, label="goal")

ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.set_title("BFS path in 3D\n(path colored blue→red by progress, obstacles in grey)")
ax.legend()
plt.tight_layout()
plt.show()

## 3D path visualization

In [ ]:
# Geodesic distance decreasing monotonically along the path — key sanity check
path_geo_dists = np.array([dist_field[v[0], v[1], v[2]] for v in path_voxels])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(path_geo_dists, color="darkorange", lw=1.5)
ax.set_xlabel("Path step")
ax.set_ylabel("Geodesic distance to goal (m)")
ax.set_title("Geodesic distance along BFS path — must be strictly decreasing")
ax.grid(True, alpha=0.3)

non_decreasing = np.sum(np.diff(path_geo_dists) >= 0)
print(f"Non-decreasing steps: {non_decreasing} / {len(path_geo_dists)-1}  (should be 0)")
print(f"Start dist: {path_geo_dists[0]:.2f} m   End dist: {path_geo_dists[-1]:.4f} m")
plt.tight_layout()
plt.show()

In [ ]:
gv = goal_voxel  # [xi, yi, zi]

# 1. Is the goal voxel itself free?
print("=== Goal voxel check ===")
print(f"Goal voxel {gv}  occupied: {occupancy[gv[0], gv[1], gv[2]]}")
print(f"Goal voxel dist: {dist_field[gv[0], gv[1], gv[2]]:.4f}  (0 = BFS started here, inf = unreachable)")

# 2. Collision check on path
collisions = [v for v in path_voxels if occupancy[v[0], v[1], v[2]]]
print(f"\n=== Path collision check ===")
print(f"Path voxels in collision: {len(collisions)} / {len(path_voxels)}  (must be 0)")

# 3. Vertical occupancy profile at goal XY — shows exactly where the wall is
print(f"\n=== Vertical profile at goal XY ({goal_world[:2]}) ===")
xi, yi = gv[0], gv[1]
occ_col = occupancy[xi, yi, :]
free_ranges = []
in_free = False
for zi, occ in enumerate(occ_col):
    z = origin[2] + zi * resolution
    if not occ and not in_free:
        start = z; in_free = True
    elif occ and in_free:
        free_ranges.append((start, z - resolution)); in_free = False
if in_free:
    free_ranges.append((start, origin[2] + (len(occ_col)-1)*resolution))
print(f"Occupied at goal XY? {occ_col.any()}  |  Free Z ranges: {[(round(a,2), round(b,2)) for a,b in free_ranges]}")
print(f"Goal Z={goal_world[2]}m — in free range? {not occupancy[gv[0], gv[1], gv[2]]}")

# 4. Show occupancy slices at 3 key heights to see where walls are vs. drone path
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
check_heights = [1.0, goal_world[2], spawn_pos[2]]  # floor, goal height, spawn height
for ax, zh in zip(axes, check_heights):
    zi = int((zh - origin[2]) / resolution)
    zi = np.clip(zi, 0, grid_shape[2] - 1)
    sl = occupancy[:, :, zi]
    ax.imshow(sl.T, origin="lower", cmap="Greys",
              extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
    ax.plot(goal_world[0], goal_world[1], "r*", ms=12, label="goal")
    ax.plot(spawn_pos[0],  spawn_pos[1],  "b^", ms=10, label="spawn")
    ax.set_title(f"Occupancy slice  z={zh}m  (zi={zi})\nOccupied: {sl.sum():,}")
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
    ax.legend()
plt.tight_layout()
plt.show()

## Diagnostic: is the goal actually free? Does the path collide?